# UT1.4 - Captura de datos desde CSV

**Módulo:** Análisis de Datos con Python

**Resultado de Aprendizaje:** RA1 - Selecciona y utiliza las librerías principales de Python para la importación, manipulación, limpieza y exportación de datos.

**Nombre del Alumno/a:** ________________________________________

**Instrucciones:** Completa los bloques `# TU CÓDIGO AQUÍ` y ejecuta la celda de test correspondiente para comprobar tu solución.

In [1]:
%reset -f
%run notebook_setup.py
%whos

Variable   Type      Data/Info
------------------------------
ipytest    module    <module 'ipytest' from '/<...>ges/ipytest/__init__.py'>
pytest     module    <module 'pytest' from '/h<...>ages/pytest/__init__.py'>


## 1. Captura y Exportación de datos en CSV (CE1.a, CE1.b, CE1.c)

El formato **CSV** (*Comma Separated Values*) es texto plano estructurado: un registro por línea y un delimitador entre columnas. Es el estándar de intercambio más extendido en la industria, pero presenta retos importantes en la práctica:
- **No almacena metadatos de tipos**: Todos los datos entran como cadenas de texto y Pandas debe inferir si son enteros, flotantes o fechas.
- **Variedad de dialectos y regiones**: En países hispanohablantes o de Europa continental, el delimitador suele ser el punto y coma (`;`) y la coma (`,`) se usa como separador decimal.
- **Codificación de caracteres (encodings)**: Los archivos generados por sistemas Windows o herramientas ofimáticas antiguas frecuentemente usan `latin-1` (ISO-8859-1) o `cp1252`, produciendo errores si se intentan abrir con `utf-8`.
- **Identificadores con ceros a la izquierda**: Códigos postales (ej. `08001`), códigos de producto, NIFs o números de teléfono pueden perder sus ceros iniciales si se infieren como números enteros (`8001`).

Pandas proporciona `pd.read_csv()` para la ingesta y `DataFrame.to_csv()` para la exportación.

### Parámetros esenciales de `pd.read_csv()`

| Parámetro | Propósito | Ejemplo de uso |
| :--- | :--- | :--- |
| `sep` / `delimiter` | Delimitador entre columnas (coma por defecto). | `sep=";"`, `sep="\t"` |
| `encoding` | Codificación del archivo de texto. | `encoding="utf-8"`, `encoding="latin-1"` |
| `decimal` | Carácter que indica el separador decimal. | `decimal=","` |
| `thousands` | Carácter para el separador de miles. | `thousands="."` |
| `dtype` | Diccionario con los tipos de datos obligatorios por columna. | `dtype={"codigo_postal": str}` |
| `usecols` | Lista de columnas a cargar (optimiza memoria). | `usecols=["producto", "precio"]` |
| `na_values` | Lista de valores de texto que deben interpretarse como `NaN`. | `na_values=["-", "N/A", "null", "DESCONOCIDO"]` |
| `keep_default_na`| Si se deben mantener los nulos por defecto (`True`/`False`). | `keep_default_na=True` |
| `parse_dates` | Lista de columnas a convertir a tipo `datetime`. | `parse_dates=["fecha"]` |
| `dayfirst` | `True` si las fechas tienen formato día-primero (`DD/MM/YYYY`). | `dayfirst=True` |
| `skiprows` | Número de líneas a saltar al principio del archivo. | `skiprows=3` |
| `comment` | Carácter que indica que el resto de la línea es un comentario. | `comment="#"` |
| `header` | Fila con los nombres de columna (`None` si no tiene cabecera). | `header=None` |
| `names` | Lista con los nombres de columna a aplicar si no hay cabecera. | `names=["id", "valor"]` |

### Parámetros esenciales de `to_csv()`

- `index=False`: Evita escribir el índice numérico de filas en una columna no deseada (`Unnamed: 0`). Casi siempre se debe fijar en `False`.
- `sep`: Delimitador de columnas deseado en el archivo de salida (ej. `sep=";"`).
- `decimal`: Separador decimal para compatibilidad (ej. `decimal=","`).
- `encoding`: Codificación de salida recomendada (`encoding="utf-8"`).

### 1.1. Ejemplo resuelto: Selección de columnas y tipado explícito

En este ejemplo creamos un archivo CSV básico y lo leemos limitando las columnas necesarias y forzando tipos para garantizar la consistencia de los datos.

In [ ]:
import pathlib
import pandas as pd

pathlib.Path("data_sample").mkdir(exist_ok=True)
ruta_demo = "data_sample/ejemplo_basico.csv"

# 1. Creamos un CSV de demostración
pd.DataFrame({
    "id": ["001", "002", "003"],
    "articulo": ["Teclado Mecánico", "Ratón Láser", "Monitor 4K"],
    "precio": [65.50, 24.90, 280.00],
    "stock": [15, 40, 8]
}).to_csv(ruta_demo, sep=";", index=False, encoding="utf-8")

# 2. Lectura selectiva con usecols y dtype
df_demo = pd.read_csv(
    ruta_demo,
    sep=";",
    usecols=["id", "articulo", "precio"],
    dtype={"id": str, "precio": float}
)

print("DataFrame cargado:")
print(df_demo)
print("\nTipos de datos:")
print(df_demo.dtypes)

Ruta actual:   /home/luis/Documentos/REPOSITORIOS/curso-26-27/271_cepy-ad/repo_publico/UT1
Ruta destino:  /home/luis/Documentos/REPOSITORIOS/curso-26-27/271_cepy-ad/repo_publico/UT1/data_sample/ejemplo_basico.csv
DataFrame cargado:
    id          articulo  precio
0  001  Teclado Mecánico    65.5
1  002       Ratón Láser    24.9
2  003        Monitor 4K   280.0

Tipos de datos:
id              str
articulo        str
precio      float64
dtype: object


### 1.2. Ejemplo resuelto: CSV europeo con decimales en coma, tildes y nulos

Muchas exportaciones de software empresarial en España vienen en `latin-1` (o `cp1252`), con `;` como separador y `,` como coma decimal.

In [ ]:
ruta_demo_eur = "data_sample/ejemplo_europeo.csv"

with open(ruta_demo_eur, "w", encoding="latin-1") as f:
    f.write("producto;precio;estado\n")
    f.write("Portátil;899,95;disponible\n")
    f.write("Móvil;249,50;N/D\n")
    f.write("Auriculares;19,99;disponible\n")

# Leemos indicando encoding, separador decimal y lista de valores nulos
df_eur = pd.read_csv(
    ruta_demo_eur,
    sep=";",
    decimal=",",
    encoding="latin-1",
    na_values=["N/D"]
)

print(df_eur)
print("\nTipo de columna precio:", df_eur["precio"].dtype)
print("Valores nulos por columna:\n", df_eur.isna().sum())

### 1.3. Generación del entorno de datos de prueba

Ejecuta la siguiente celda para generar en la carpeta `data_sample/` los archivos CSV necesarios para resolver la batería de ejercicios.

In [ ]:
import pathlib
import pandas as pd

pathlib.Path("data_sample").mkdir(exist_ok=True)

# 1. ventas_europa.csv (latin-1, sep=';', decimal=',', miles='.')
with open("data_sample/ventas_europa.csv", "w", encoding="latin-1") as f:
    f.write(
        "id_producto;producto;unidades;precio_unitario;total\n"
        "1;Portátil Profesional;2;1.250,50;2.501,00\n"
        "2;Ratón Óptico;10;25,75;257,50\n"
        "3;Cámara Web HD;5;45,00;225,00\n"
        "4;Impresora Multifunción;1;350,25;350,25\n"
    )

# 2. clientes_padron.csv (identificadores con ceros a la izquierda y nulos sucios)
with open("data_sample/clientes_padron.csv", "w", encoding="utf-8") as f:
    f.write(
        "id_cliente,nombre,codigo_postal,email,edad\n"
        "001,Elena Gómez,08001,elena@example.com,34\n"
        "002,Carlos Ruiz,01001,-,45\n"
        "003,Marta Sáez,03002,marta@example.com,N/A\n"
        "004,Javier Gil,28001,DESCONOCIDO,29\n"
        "005,Lucía Ramos,02004,lucia@example.com,null\n"
    )

# 3. sensor_metadatos.csv (metadatos iniciales, comentarios '#' y sin cabecera)
with open("data_sample/sensor_metadatos.csv", "w", encoding="utf-8") as f:
    f.write(
        "# REGISTRO DE DATOS IOT - VERSION 2.1\n"
        "# Dispositivo: Sensor-Estacion-Norte\n"
        "# Formato: timestamp,temperatura,humedad,presion\n"
        "2026-05-01 08:00,21.5,60.2,1013.2\n"
        "# CALIBRACION EN CURSO\n"
        "2026-05-01 09:00,22.1,58.4,1012.8\n"
        "2026-05-01 10:00,23.8,55.0,1012.5\n"
        "# REVISION COMPLETADA\n"
        "2026-05-01 11:00,24.5,53.8,1011.9\n"
    )

# 4. transacciones_fechas.csv (fechas en formato europeo DD/MM/YYYY)
with open("data_sample/transacciones_fechas.csv", "w", encoding="utf-8") as f:
    f.write(
        "id_transaccion;fecha;cliente;precio\n"
        "TX101;05/02/2024;TechCorp;1500.00\n"
        "TX102;12/07/2024;BioHealth;2300.50\n"
        "TX103;20/11/2024;GlobalLogistics;850.00\n"
        "TX104;15/01/2025;TechCorp;3200.00\n"
        "TX105;28/04/2025;RetailPlus;450.75\n"
    )

# 5. ventas_tiendas.csv (ventas por categoría y tienda para agregaciones)
with open("data_sample/ventas_tiendas.csv", "w", encoding="utf-8") as f:
    f.write(
        "id_venta,tienda,categoria,importe\n"
        "1,Madrid,Electrónica,1200.00\n"
        "2,Barcelona,Hogar,350.50\n"
        "3,Madrid,Hogar,150.00\n"
        "4,Valencia,Electrónica,800.00\n"
        "5,Sevilla,Ropa,120.00\n"
        "6,Madrid,Electrónica,600.00\n"
        "7,Barcelona,Ropa,250.00\n"
        "8,Valencia,Hogar,420.50\n"
    )

print("✅ Archivos de datos de prueba creados en data_sample/")

## 2. Batería de Ejercicios Prácticos

#### Ejercicio 1.1: Ingesta Europea y Formato Numérico (CE1.a, CE1.b)

El fichero `data_sample/ventas_europa.csv` contiene transacciones comerciales exportadas desde un sistema administrativo en España:
- Está codificado en `latin-1` (contiene tildes como *Portátil* o *Ratón*).
- Utiliza el punto y coma (`;`) como separador de campos.
- Utiliza la coma (`,`) como separador decimal y el punto (`.`) como separador de miles.

Implementa la función `leer_ventas_europa(ruta: str) -> pd.DataFrame` que lea el fichero configurando adecuadamente los parámetros de `read_csv()` para que:
1. Las tildes y caracteres se carguen correctamente.
2. Las columnas `precio_unitario` y `total` sean de tipo `float64`.

In [ ]:
def leer_ventas_europa(ruta: str) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa leer_ventas_europa")

In [ ]:
# TEST EJERCICIO 1.1
import numpy as np

df_ventas_eur = leer_ventas_europa("data_sample/ventas_europa.csv")

assert list(df_ventas_eur.columns) == ["id_producto", "producto", "unidades", "precio_unitario", "total"]
assert df_ventas_eur["precio_unitario"].dtype == "float64", "precio_unitario debe ser de tipo flotante"
assert df_ventas_eur["total"].dtype == "float64", "total debe ser de tipo flotante"
assert np.isclose(df_ventas_eur["total"].sum(), 3333.75), "La suma total de importes no coincide"
assert "Portátil Profesional" in df_ventas_eur["producto"].values, "Error en la codificación de caracteres"
assert "Ratón Óptico" in df_ventas_eur["producto"].values, "Error en la codificación de caracteres"
print("✅ Test Ejercicio 1.1 superado.")

#### Ejercicio 1.2: Preservación de Ceros a la Izquierda y Nulos Personalizados (CE1.b)

El fichero `data_sample/clientes_padron.csv` contiene registros de usuarios con códigos postales y números de identificación que comienzan por `0` (ej. `08001`, `01001`, `001`). Además, la columna `email` y `edad` presentan valores nulos introducidos con cadenas heterogéneas como `"-"`, `"N/A"`, `"null"` y `"DESCONOCIDO"`.

Implementa la función `leer_clientes_padron(ruta: str) -> pd.DataFrame` que:
1. Fuerce las columnas `id_cliente` y `codigo_postal` a ser de tipo texto (`str` o `object`) para no truncar los ceros a la izquierda.
2. Convierta en valores nulos (`NaN`) las cadenas `"-"`, `"N/A"`, `"null"` y `"DESCONOCIDO"`.

In [ ]:
def leer_clientes_padron(ruta: str) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa leer_clientes_padron")

In [ ]:
# TEST EJERCICIO 1.2
df_padron = leer_clientes_padron("data_sample/clientes_padron.csv")

assert df_padron["codigo_postal"].dtype == "object", "codigo_postal debe ser de tipo object/string"
assert df_padron["id_cliente"].dtype == "object", "id_cliente debe ser de tipo object/string"
assert df_padron.loc[df_padron["nombre"] == "Elena Gómez", "codigo_postal"].values[0] == "08001", "El código postal 08001 no conservó el cero inicial"
assert df_padron.loc[df_padron["nombre"] == "Carlos Ruiz", "codigo_postal"].values[0] == "01001", "El código postal 01001 no conservó el cero inicial"
assert df_padron.loc[df_padron["nombre"] == "Elena Gómez", "id_cliente"].values[0] == "001", "El identificador 001 perdió sus ceros"
assert df_padron["email"].isna().sum() == 2, "email debe tener exactamente 2 valores nulos detectados"
assert df_padron["edad"].isna().sum() == 2, "edad debe tener exactamente 2 valores nulos detectados"
print("✅ Test Ejercicio 1.2 superado.")

#### Ejercicio 1.3: Metadatos, Comentarios y Cabeceras Dinámicas (CE1.b)

El fichero `data_sample/sensor_metadatos.csv` es un volcado directo de sensores industriales que:
- Contiene metadatos en las primeras líneas y líneas comentadas con `#` intercaladas.
- No incluye fila de cabecera con nombres de columnas en los datos.

Implementa la función `leer_datos_sensor(ruta: str, columnas: list[str]) -> pd.DataFrame` que:
1. Ignore las líneas que comiencen por `#` como comentarios.
2. No tome la primera línea de datos como cabecera (`header=None`).
3. Asigne la lista `columnas` como nombres de las columnas del DataFrame.

In [ ]:
def leer_datos_sensor(ruta: str, columnas: list[str]) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa leer_datos_sensor")

In [ ]:
# TEST EJERCICIO 1.3
columnas_esperadas = ["timestamp", "temperatura", "humedad", "presion"]
df_sensor = leer_datos_sensor("data_sample/sensor_metadatos.csv", columnas_esperadas)

assert list(df_sensor.columns) == columnas_esperadas, "Los nombres de columna no coinciden con la lista proporcionada"
assert len(df_sensor) == 4, f"Se esperaban 4 filas de datos útiles, se obtuvieron {len(df_sensor)}"
assert df_sensor["temperatura"].dtype == "float64", "La temperatura debe ser numérica flotante"
assert df_sensor.iloc[0]["timestamp"] == "2026-05-01 08:00"
assert df_sensor.iloc[-1]["temperatura"] == 24.5
print("✅ Test Ejercicio 1.3 superado.")

#### Ejercicio 1.4: Fechas y Exportación manteniendo estructura (CE1.b, CE1.c)

El fichero `data_sample/transacciones_fechas.csv` contiene transacciones comerciales con delimitador `;` y fechas en formato día primero (`DD/MM/YYYY`).

Implementa la función `procesar_y_exportar_transacciones(ruta_origen: str, ruta_destino: str, anio: int) -> int` que:
1. Lea el CSV parseando la columna `fecha` como tipo `datetime` (`parse_dates=["fecha"]`, `dayfirst=True`).
2. Filtre las transacciones cuyo año (`fecha.dt.year`) coincida con el argumento `anio`.
3. Cree una nueva columna llamada `precio_con_iva` calculada como `precio * 1.21` redondeada a 2 decimales.
4. Guarde el DataFrame resultante en `ruta_destino` como un archivo CSV con separador `;`, codificación `utf-8` y **sin índice** (`index=False`).
5. Devuelva la cantidad de filas exportadas (número entero).

In [ ]:
def procesar_y_exportar_transacciones(ruta_origen: str, ruta_destino: str, anio: int) -> int:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa procesar_y_exportar_transacciones")

In [ ]:
# TEST EJERCICIO 1.4
import numpy as np

ruta_dest_4 = "data_sample/transacciones_2024.csv"
num_filas = procesar_y_exportar_transacciones("data_sample/transacciones_fechas.csv", ruta_dest_4, anio=2024)

assert num_filas == 3, f"Se esperaban 3 filas para el año 2024, se devolvieron {num_filas}"

# Validar el archivo CSV exportado
df_verif_4 = pd.read_csv(ruta_dest_4, sep=";", parse_dates=["fecha"])
assert len(df_verif_4) == 3
assert list(df_verif_4["id_transaccion"]) == ["TX101", "TX102", "TX103"]
assert "precio_con_iva" in df_verif_4.columns, "Falta la columna precio_con_iva"
assert np.isclose(df_verif_4.loc[df_verif_4["id_transaccion"] == "TX101", "precio_con_iva"].values[0], 1815.00)
assert "Unnamed: 0" not in df_verif_4.columns, "El CSV exportado no debe contener columna de índice residual"
print("✅ Test Ejercicio 1.4 superado.")

#### Ejercicio 1.5: Agregación, Cambio de estructura y Exportación Regional (CE1.b, CE1.c, CE1.d)

El fichero `data_sample/ventas_tiendas.csv` contiene registros de ventas detalladas por categoría y tienda.

Implementa la función `generar_resumen_ventas(ruta_origen: str, ruta_destino: str) -> pd.DataFrame` que:
1. Lea el CSV de origen.
2. Agrupe por la columna `categoria` y calcule las siguientes métricas:
   - `ingresos_totales`: suma de `importe`.
   - `ticket_medio`: media de `importe` redondeada a 2 decimales.
   - `num_operaciones`: recuento de operaciones (`count` sobre `importe`).
3. Convierta el resultado en un DataFrame plano con `categoria` como columna ordinaria (`reset_index()`).
4. Guarde el resumen en `ruta_destino` en formato regional europeo: con delimitador punto y coma (`sep=";"`), coma como separador decimal (`decimal=","`), sin índice (`index=False`) y con codificación `utf-8`.
5. Devuelva el DataFrame resultante.

In [ ]:
def generar_resumen_ventas(ruta_origen: str, ruta_destino: str) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa generar_resumen_ventas")

In [ ]:
# TEST EJERCICIO 1.5
import numpy as np

ruta_dest_5 = "data_sample/resumen_categorias.csv"
df_resumen = generar_resumen_ventas("data_sample/ventas_tiendas.csv", ruta_dest_5)

assert list(df_resumen.columns) == ["categoria", "ingresos_totales", "ticket_medio", "num_operaciones"], "Columnas del resumen incorrectas"
assert len(df_resumen) == 3, f"Se esperaban 3 categorías, se obtuvieron {len(df_resumen)}"

row_elec = df_resumen[df_resumen["categoria"] == "Electrónica"]
assert row_elec["ingresos_totales"].values[0] == 2600.00
assert row_elec["num_operaciones"].values[0] == 3
assert np.isclose(row_elec["ticket_medio"].values[0], 866.67)

# Validar que el CSV exportado cumple con el formato regional europeo
with open(ruta_dest_5, "r", encoding="utf-8") as f:
    lineas = f.readlines()

assert "categoria;ingresos_totales;ticket_medio;num_operaciones\n" in lineas[0], "La cabecera debe usar ';' como separador"
assert any("," in linea for linea in lineas[1:]), "El archivo exportado debe usar la coma como separador decimal"

# Lectura de verificación con Pandas usando sep=';' y decimal=','
df_leido = pd.read_csv(ruta_dest_5, sep=";", decimal=",")
assert len(df_leido) == 3
assert df_leido["ticket_medio"].dtype == "float64"
assert np.isclose(df_leido.loc[df_leido["categoria"] == "Electrónica", "ticket_medio"].values[0], 866.67)
print("✅ Test Ejercicio 1.5 superado.")